任务目标
把下面四件事真正连起来：
- requires_grad=True：记录后续运算，以便求导。
- .backward()：从结果出发，反向计算梯度。
- .grad：查看默认保存到叶子张量上的梯度。
- .retain_grad()：让非叶子张量也保存梯度。

---

In [ ]:
"""
第一题：标量求导
补全代码中两处 TODO：
"""

import torch

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x

print("Before backward:", x.grad)

# TODO: 对 y 进行反向传播
y.backward()

print("y:", y)
print("x.grad:", x.grad)

# TODO: 写出手算导数，并验证 x.grad 是否正确
assert torch.allclose(x.grad, torch.tensor(8.0))

---

In [ ]:
"""
第二题：向量输入，标量输出
"""

v = torch.tensor([1.0, -2.0, 3.0], requires_grad=True)
loss = (v ** 2).sum()

# TODO: 反向传播
loss.backward()

print("loss:", loss)
print("v.grad:", v.grad)
print("v.grad.shape:", v.grad.shape)

# TODO: 用 assert 验证梯度
assert torch.allclose(v.grad, torch.tensor([2.0, -4.0, 6.0]))

---

In [ ]:
"""
第三题：观察叶子与非叶子
"""

a = torch.tensor(2.0, requires_grad=True)
u = a ** 2
z = 3 * u

for name, t in [("a", a), ("u", u), ("z", z)]:
    print(
        name,
        "requires_grad:", t.requires_grad,
        "is_leaf:", t.is_leaf,
        "grad_fn:", t.grad_fn,
    )

z.backward()

print("a.grad:", a.grad)
print("u.grad:", u.grad)

1. 哪个张量是叶子？\
    a
2. 哪些张量的 grad_fn 不是 None？\
    u, z
3. u.grad 是什么？这是否表示 \(\partial z/\partial u=0\)？\
    u.grad 为 None，表示本次反向传播没有把梯度保存在 u.grad 中，因为 u 是非叶子张量且没有调用 retain_grad()。这不表示导数为零。

---

In [ ]:
"""
第四题：保留中间梯度
"""

a = torch.tensor(2.0, requires_grad=True)
u = a ** 2
z = 3 * u

for name, t in [("a", a), ("u", u), ("z", z)]:
    print(
        name,
        "requires_grad:", t.requires_grad,
        "is_leaf:", t.is_leaf,
        "grad_fn:", t.grad_fn,
    )

u.retain_grad()

z.backward()

print("a.grad:", a.grad)
print("u.grad:", u.grad)

# dz/du = 3
# dz/da = (dz/du) * (du/da) = 3 * (2*a) = 12
assert torch.allclose(u.grad, torch.tensor(3.0))
assert torch.allclose(a.grad, torch.tensor(12.0))